In [18]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l
from torchinfo import summary

In [8]:
class Residual(nn.Module):
    def __init__(self, input_channels,num_channels,use_1x1conv=False,strides=1):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels,num_channels,kernel_size=3,padding=1,stride=strides)
        self.conv2 = nn.Conv2d(num_channels,num_channels,kernel_size=3,padding=1)
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels,num_channels,kernel_size=1,stride=strides)
        else:
            self.conv3 = None
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)

    def forward(self,x):
        Y = F.relu(self.bn1(self.conv1(x)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            x = self.conv3(x)
        Y += x
        return F.relu(Y)

'''
卷积维度计算公式：
输出尺寸 = floor((输入尺寸 - kernel_size + 2*padding) / stride) + 1
代入计算：(6 - 3 + 2*1)/1 + 1 = 6  一般来说正方形尺寸
所以 self.conv1(X) 的维度：(4, 3, 6, 6)
后续经过 bn1 + relu 后，维度不变：(4, 3, 6, 6)
'''

'\n卷积维度计算公式：\n输出尺寸 = floor((输入尺寸 - kernel_size + 2*padding) / stride) + 1\n代入计算：(6 - 3 + 2*1)/1 + 1 = 6  一般来说正方形尺寸\n所以 self.conv1(X) 的维度：(4, 3, 6, 6)\n后续经过 bn1 + relu 后，维度不变：(4, 3, 6, 6)\n'

In [9]:
blk = Residual(3,3)
X = torch.randn(4,3,6,6)
Y = blk(X)
Y.shape

torch.Size([4, 3, 6, 6])

In [12]:
blk = Residual(3,6,use_1x1conv=True,strides=2)
X = torch.randn(4,3,6,6)
'''
1x1 卷积的核心作用是：当主分支的通道 / 尺寸变化时，让残差分支（原始 x）的维度同步变化，保证 Y += x 可以执行
'''
blk(X).shape

torch.Size([4, 6, 3, 3])

In [13]:
b1 = nn.Sequential(nn.Conv2d(1,64,kernel_size=7,stride=2,padding=3),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

In [14]:
def resnet_block(input_channels,num_channels,num_residuals,first_block=False):
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.append(Residual(input_channels,num_channels,use_1x1conv=True,strides=2))
        else:
            blk.append(Residual(num_channels,num_channels))
    return blk

In [16]:
b2 = nn.Sequential(*resnet_block(64,64,2,first_block=True))  # 拆包
b3 = nn.Sequential(*resnet_block(64,128,2))
b4 = nn.Sequential(*resnet_block(128,256,2))
b5 = nn.Sequential(*resnet_block(256,512,2))
net = nn.Sequential(b1,b2,b3,b4,b5,nn.AdaptiveAvgPool2d((1,1)),nn.Flatten(),nn.Linear(512,10))
# 直接指定输出的尺寸，PyTorch 会自动计算需要的核大小 / 步长，保证输出是想要的尺寸
# 输出的空间尺寸都会被池化成 1×1，通道数和批次大小保持不变

In [25]:
summary(net,input_size=(1,1,224,224))

Layer (type:depth-idx)                   Output Shape              Param #
Sequential                               [1, 10]                   --
├─Sequential: 1-1                        [1, 64, 56, 56]           --
│    └─Conv2d: 2-1                       [1, 64, 112, 112]         3,200
│    └─BatchNorm2d: 2-2                  [1, 64, 112, 112]         128
│    └─ReLU: 2-3                         [1, 64, 112, 112]         --
│    └─MaxPool2d: 2-4                    [1, 64, 56, 56]           --
├─Sequential: 1-2                        [1, 64, 56, 56]           --
│    └─Residual: 2-5                     [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-1                  [1, 64, 56, 56]           36,928
│    │    └─BatchNorm2d: 3-2             [1, 64, 56, 56]           128
│    │    └─Conv2d: 3-3                  [1, 64, 56, 56]           36,928
│    │    └─BatchNorm2d: 3-4             [1, 64, 56, 56]           128
│    └─Residual: 2-6                     [1, 64, 56, 56]           --
│

In [ ]:
lr, num_epochs, batch_size = 0.05,10,256
train_iter,test_iter = d2l.load_data_fashion_mnist(batch_size, resize=96)
d2l.train_ch3(net,train_iter,test_iter,num_epochs,lr,d2l.try_gpu())